# 19A4E2 — 2024 vs 2025 AIA Input / Production-Shift Audit

## Purpose

Follow up the frozen post-test score-shift finding by testing whether the **input AIA tensors themselves** changed between 2024 and 2025.

The frozen 19A4 result is not modified.

This notebook compares a deterministic sample of unique AIA objects from 2024 and 2025 and records:

- NPZ key/schema frequencies;
- source / channel metadata;
- raw per-channel pixel statistics;
- frozen-preprocessing transformed per-channel statistics;
- non-finite and zero fractions;
- object-level distribution summaries;
- 2024-vs-2025 distribution tests.

### Scientific interpretation rule

A detected input distribution change establishes **covariate / production shift**, but it does not by itself prove whether the cause is:
- solar-regime change,
- acquisition / preprocessing change,
- production-pipeline change,
- or a mixture.

No model weights, calibrator, threshold, or primary Cycle-25 score are altered.


In [1]:
from pathlib import Path
import json, os, shutil, subprocess, tempfile
from collections import Counter
import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

HOME = Path.home()
PREP = HOME / "aia19_cycle25_staging_prep"
OUT = HOME / "aia19_cycle25_input_shift_audit"
OUT.mkdir(parents=True, exist_ok=True)

NORM_PATH = HOME / "aia19_cnn_gru_20260917" / "normalisation.json"
assert NORM_PATH.exists(), NORM_PATH

EXPECTED_CHANNELS = ["aia94","aia131","aia171","aia193","aia211","aia335"]
EXPECTED_WAVELENGTHS = [94,131,171,193,211,335]

YEARS = [2024, 2025]
SAMPLE_OBJECTS_PER_YEAR = 400
PIXELS_PER_OBJECT = 2048
SEED = 20260919

rng = np.random.default_rng(SEED)

norm = json.loads(NORM_PATH.read_text())
scale = np.asarray(norm["channel_scale"], dtype=np.float32)
mean = np.asarray(norm["channel_mean"], dtype=np.float32)
std = np.asarray(norm["channel_std"], dtype=np.float32)

print("Sample objects/year:", SAMPLE_OBJECTS_PER_YEAR)
print("Pixels/object/channel:", PIXELS_PER_OBJECT)


Sample objects/year: 400
Pixels/object/channel: 2048


## 1. Select deterministic unique-object samples

2025 can be read from the local staged cache if present.  
2024 objects are fetched on demand from their original GCS URIs if the disposable 2024 cache has already been removed.


In [2]:
samples = {}

for year in YEARS:
    inv = PREP / f"cycle25_{year}_unique_aia_objects.csv.gz"
    assert inv.exists(), inv

    d = pd.read_csv(inv)
    assert "object_uri" in d.columns

    n = min(SAMPLE_OBJECTS_PER_YEAR, len(d))
    idx = rng.choice(len(d), size=n, replace=False)
    s = d.iloc[np.sort(idx)].copy().reset_index(drop=True)

    samples[year] = s
    print(year, "inventory:", len(d), "sample:", len(s))


2024 inventory: 13831 sample: 400
2025 inventory: 10879 sample: 400


In [3]:
def local_existing_path(year, uri):
    filename = uri.rsplit("/", 1)[-1]
    p = Path("/mnt/disks/aia-cache/cycle25_yearwise") / str(year) / filename
    return p if p.exists() else None

def fetch_to_temp(uri, temp_dir):
    dest = Path(temp_dir) / uri.rsplit("/", 1)[-1]
    subprocess.run(
        ["gcloud", "storage", "cp", "--quiet", uri, str(dest)],
        check=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )
    return dest

def metadata_contract(z):
    if "channels" in z.files:
        ch = [str(v) for v in z["channels"].tolist()]
        return "channels", ch
    if "wavelengths" in z.files:
        wl = [int(v) for v in z["wavelengths"].tolist()]
        return "wavelengths", wl
    return "none", None

def scalar_or_repr(z, key):
    if key not in z.files:
        return None
    try:
        v = z[key]
        if np.ndim(v) == 0:
            return v.item()
        return v.tolist()
    except Exception:
        return repr(z[key])


## 2. Audit schemas and pixel distributions

In [4]:
object_rows = []
pixel_samples = {year: [[] for _ in range(6)] for year in YEARS}
transformed_samples = {year: [[] for _ in range(6)] for year in YEARS}
schema_counts = {year: Counter() for year in YEARS}

with tempfile.TemporaryDirectory(prefix="aia_2024_2025_shift_") as td:
    td = Path(td)

    for year in YEARS:
        print(f"\n===== YEAR {year} =====", flush=True)

        for i, r in samples[year].iterrows():
            uri = r["object_uri"]

            local = local_existing_path(year, uri)
            fetched = False

            if local is None:
                local = fetch_to_temp(uri, td)
                fetched = True

            with np.load(local, allow_pickle=False) as z:
                schema = tuple(sorted(z.files))
                schema_counts[year][schema] += 1

                if "x" not in z.files:
                    raise RuntimeError(f"No x tensor: {uri}")

                x = z["x"]
                if x.shape != (512,512,6):
                    raise RuntimeError(f"Bad shape {x.shape}: {uri}")
                if x.dtype != np.float32:
                    raise RuntimeError(f"Bad dtype {x.dtype}: {uri}")
                if not np.isfinite(x).all():
                    raise RuntimeError(f"Nonfinite tensor: {uri}")

                metadata_mode, metadata_value = metadata_contract(z)

                if metadata_mode == "channels":
                    assert metadata_value == EXPECTED_CHANNELS
                elif metadata_mode == "wavelengths":
                    assert metadata_value == EXPECTED_WAVELENGTHS
                else:
                    raise RuntimeError(f"No channel-order metadata: {uri}")

                flat = x.reshape(-1, 6)
                take = rng.choice(
                    flat.shape[0],
                    size=min(PIXELS_PER_OBJECT, flat.shape[0]),
                    replace=False,
                )
                pix = flat[take].astype(np.float64, copy=False)

                transformed = np.arcsinh(
                    pix / scale.reshape(1,6)
                )
                transformed = (
                    transformed - mean.reshape(1,6)
                ) / std.reshape(1,6)

                rec = {
                    "year": year,
                    "uri": uri,
                    "metadata_mode": metadata_mode,
                    "schema": "|".join(schema),
                    "source": json.dumps(scalar_or_repr(z, "source"), default=str),
                    "channel_metadata": json.dumps(
                        scalar_or_repr(z, "channel_metadata"), default=str
                    ),
                    "block_id": json.dumps(scalar_or_repr(z, "block_id"), default=str),
                    "raw_global_mean": float(np.mean(x)),
                    "raw_global_std": float(np.std(x)),
                    "raw_zero_fraction": float(np.mean(x == 0)),
                    "raw_abs_q99": float(np.quantile(np.abs(x), .99)),
                }

                for c in range(6):
                    v = pix[:, c]
                    tv = transformed[:, c]

                    pixel_samples[year][c].append(v)
                    transformed_samples[year][c].append(tv)

                    rec[f"ch{c}_raw_median"] = float(np.median(v))
                    rec[f"ch{c}_raw_q99"] = float(np.quantile(v, .99))
                    rec[f"ch{c}_transformed_median"] = float(np.median(tv))
                    rec[f"ch{c}_transformed_q99"] = float(np.quantile(tv, .99))

                object_rows.append(rec)

            if fetched:
                local.unlink(missing_ok=True)

            if (i + 1) % 50 == 0 or (i + 1) == len(samples[year]):
                print(f"{year}: {i+1}/{len(samples[year])}", flush=True)

objects = pd.DataFrame(object_rows)
objects.to_csv(OUT / "object_level_input_statistics.csv.gz", index=False, compression="gzip")

print("\nSchema counts:")
for year in YEARS:
    print("\nYEAR", year)
    for schema, n in schema_counts[year].most_common():
        print(n, schema)



===== YEAR 2024 =====


2024: 50/400


2024: 100/400


2024: 150/400


2024: 200/400


...

2025: 250/400


2025: 300/400


2025: 350/400


2025: 400/400



Schema counts:

YEAR 2024
400 ('HARPNUM', 'NOAA_AR_clean', 'T_REC_dt', 'channels', 'crop_meta', 'sample_id', 'used_s3_path', 'used_timestamp', 'x', 'y')

YEAR 2025
400 ('HARPNUM', 'NOAA_AR_clean', 'T_REC_dt', 'block_id', 'channel_metadata', 'label_48h_final', 'sample_id', 'source', 'wavelengths', 'x', 'y')


## 3. Channel-level 2024 vs 2025 comparison

In [5]:
channel_rows = []
ks_rows = []

for c, name in enumerate(EXPECTED_CHANNELS):
    a_raw = np.concatenate(pixel_samples[2024][c])
    b_raw = np.concatenate(pixel_samples[2025][c])

    a_tx = np.concatenate(transformed_samples[2024][c])
    b_tx = np.concatenate(transformed_samples[2025][c])

    for year, vals_raw, vals_tx in [
        (2024, a_raw, a_tx),
        (2025, b_raw, b_tx),
    ]:
        channel_rows.append({
            "year": year,
            "channel_index": c,
            "channel": name,
            "n_pixels": int(len(vals_raw)),
            "raw_mean": float(np.mean(vals_raw)),
            "raw_std": float(np.std(vals_raw)),
            "raw_q01": float(np.quantile(vals_raw, .01)),
            "raw_q25": float(np.quantile(vals_raw, .25)),
            "raw_q50": float(np.quantile(vals_raw, .50)),
            "raw_q75": float(np.quantile(vals_raw, .75)),
            "raw_q99": float(np.quantile(vals_raw, .99)),
            "raw_zero_fraction": float(np.mean(vals_raw == 0)),
            "transformed_mean": float(np.mean(vals_tx)),
            "transformed_std": float(np.std(vals_tx)),
            "transformed_q01": float(np.quantile(vals_tx, .01)),
            "transformed_q50": float(np.quantile(vals_tx, .50)),
            "transformed_q99": float(np.quantile(vals_tx, .99)),
        })

    for var, aa, bb in [
        ("raw", a_raw, b_raw),
        ("frozen_preprocessed", a_tx, b_tx),
    ]:
        stat, p = ks_2samp(aa, bb)
        ks_rows.append({
            "channel_index": c,
            "channel": name,
            "variable": var,
            "ks_statistic": float(stat),
            "p_value": float(p),
            "n_2024": int(len(aa)),
            "n_2025": int(len(bb)),
        })

channel_summary = pd.DataFrame(channel_rows)
channel_ks = pd.DataFrame(ks_rows)

channel_summary.to_csv(OUT / "channel_distribution_summary.csv", index=False)
channel_ks.to_csv(OUT / "channel_ks_2024_vs_2025.csv", index=False)

print(channel_summary.to_string(index=False))
print("\nKS:")
print(channel_ks.to_string(index=False))


 year  channel_index channel  n_pixels  raw_mean  raw_std  raw_q01  raw_q25  raw_q50  raw_q75  raw_q99  raw_zero_fraction  transformed_mean  transformed_std  transformed_q01  transformed_q50  transformed_q99
 2024              0   aia94    819200  0.312039 0.153179 0.061825 0.194262 0.292199 0.409018 0.731715           0.000076          0.292750         1.065483        -1.734050         0.258913         2.723950
 2025              0   aia94    819200  0.410531 0.140344 0.113585 0.315381 0.403190 0.497115 0.776386           0.003259          0.988027         0.889059        -1.244450         1.026451         2.906822
 2024              1  aia131    819200  0.373316 0.142551 0.107852 0.269487 0.357400 0.461926 0.755241           0.000081          0.188460         1.051469        -2.052987         0.151573         2.623602
 2025              1  aia131    819200  0.419586 0.140347 0.124735 0.323951 0.406686 0.503603 0.791316           0.003278          0.536407         0.996721        -1.8

## 4. Production metadata summary

In [6]:
meta_summary = (
    objects.groupby(["year","metadata_mode","schema","source","channel_metadata"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["year","count"], ascending=[True,False])
)

meta_summary.to_csv(OUT / "production_metadata_summary.csv", index=False)

print(meta_summary.head(30).to_string(index=False))


 year metadata_mode                                                                                                    schema                                              source                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

## 5. Diagnostic record

In [7]:
diagnostic = {
    "status": "POST_TEST_2024_2025_AIA_INPUT_SHIFT_AUDIT_COMPLETE_NO_TUNING",
    "sample_objects_per_year": SAMPLE_OBJECTS_PER_YEAR,
    "pixels_per_object": PIXELS_PER_OBJECT,
    "seed": SEED,
    "years": YEARS,
    "frozen_model_changed": False,
    "frozen_normalisation_changed": False,
    "calibrator_changed": False,
    "threshold_changed": False,
    "primary_cycle25_metrics_changed": False,
    "outputs": [
        "object_level_input_statistics.csv.gz",
        "channel_distribution_summary.csv",
        "channel_ks_2024_vs_2025.csv",
        "production_metadata_summary.csv",
    ],
    "interpretation_constraint": (
        "Observed input-distribution differences establish covariate/production "
        "shift but do not by themselves identify physical cause."
    ),
}

(OUT / "diagnostic_record.json").write_text(json.dumps(diagnostic, indent=2) + "\n")

print(json.dumps(diagnostic, indent=2))
print("\n19A4E2_INPUT_SHIFT_AUDIT_COMPLETE")


{
  "status": "POST_TEST_2024_2025_AIA_INPUT_SHIFT_AUDIT_COMPLETE_NO_TUNING",
  "sample_objects_per_year": 400,
  "pixels_per_object": 2048,
  "seed": 20260919,
  "years": [
    2024,
    2025
  ],
  "frozen_model_changed": false,
  "frozen_normalisation_changed": false,
  "calibrator_changed": false,
  "threshold_changed": false,
  "primary_cycle25_metrics_changed": false,
  "outputs": [
    "object_level_input_statistics.csv.gz",
    "channel_distribution_summary.csv",
    "channel_ks_2024_vs_2025.csv",
    "production_metadata_summary.csv"
  ],
  "interpretation_constraint": "Observed input-distribution differences establish covariate/production shift but do not by themselves identify physical cause."
}

19A4E2_INPUT_SHIFT_AUDIT_COMPLETE
